# Hotel Booking Cancellation Prediction – Random Forest

**Prerequisite:** run `FDM_Preprocessing_KNN_RF.ipynb` (Parts A & B) first. It writes the preprocessed matrices to `data/` and the fitted encoders to `models/preprocessors.joblib`.

Trees do not need scaling or winsorising, so the forest uses the **tree** feature matrix (`X_*_tree.csv`): raw numerics, the ordinal lead-time band and one-hot categories, with encoders fitted on the training set only.

**Validation protocol (from the proposal):**
- Cross-validation uses `StratifiedGroupKFold` with the same predictor-hash groups as the train/test split, so identical bookings never sit on both sides of a fold.
- Hyper-parameters are tuned with a **randomised search on the full training set**. An earlier version tuned on a 25% sample; the best settings depend on data size (more data supports deeper trees), so tuning on all rows is one of the optimisations measured below.
- Imbalance is handled with `class_weight`. `None`, `balanced` and `balanced_subsample` are compared in the search.
- Tuning is scored by **PR-AUC**; recall, precision, F1, ROC-AUC and confusion matrices are also reported.
- The decision threshold is chosen from **out-of-fold training predictions** using a stated cost assumption. The test set is only used for the final evaluation.

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, precision_recall_curve,
                             classification_report)
from sklearn.inspection import permutation_importance

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
N_JOBS = -1

### LOAD PREPROCESSED DATA

In [ ]:
X_train_tree = pd.read_csv("data/X_train_tree.csv")
X_test_tree = pd.read_csv("data/X_test_tree.csv")
y_train = pd.read_csv("data/y_train.csv").squeeze("columns").astype(int)
y_test = pd.read_csv("data/y_test.csv").squeeze("columns").astype(int)
meta_train = pd.read_csv("data/meta_train.csv", parse_dates=["arrival_date"])
meta_test = pd.read_csv("data/meta_test.csv", parse_dates=["arrival_date"])

train_groups_arr = meta_train["group"].to_numpy()
print("X_train_tree:", X_train_tree.shape, "| X_test_tree:", X_test_tree.shape)
print(f"Cancellation rate - train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")

### HELPERS – metrics, CV splitter

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def evaluate(y_true, proba, threshold=0.5):
    """Standard metric set for the cancellation (positive) class."""
    pred = (proba >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "ROC-AUC": roc_auc_score(y_true, proba),
        "PR-AUC": average_precision_score(y_true, proba),
    }

def oof_rf(params, X, y, groups):
    """Out-of-fold probabilities for a Random Forest with the given parameters."""
    oof = np.zeros(len(y))
    for tr, va in cv.split(X, y, groups=groups):
        m = RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=N_JOBS)
        m.fit(X.iloc[tr], y.iloc[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
    return oof

### BASELINE RANDOM FOREST (sklearn defaults)

In [ ]:
t0 = time.time()
rf_base_oof = oof_rf({}, X_train_tree, y_train, train_groups_arr)
rf_baseline_cv = evaluate(y_train, rf_base_oof)
print(f"RF baseline CV ({time.time() - t0:.1f}s):")
print(pd.Series(rf_baseline_cv).round(4).to_string())

### HYPER-PARAMETER TUNING (randomised search with group-aware 5-fold CV, full training set)

The search uses 200 trees per forest to keep it fast; the number of trees is not tuned because more trees only make the average steadier. The final model uses 400. Parallelism is inside each forest (`n_jobs=-1` on the model, `n_jobs=1` on the search), which avoids copying the data to many worker processes.

In [ ]:
rf_param_dist = {
    "max_depth": [None, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.2, 0.3, 0.5],
    "class_weight": [None, "balanced", "balanced_subsample"],
    "max_samples": [None, 0.7],          # share of rows each tree is grown on
    "criterion": ["gini", "entropy"],    # how split quality is measured
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=N_JOBS),
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc", "f1": "f1", "accuracy": "accuracy",
             "recall": "recall", "precision": "precision"},
    refit=False,
    cv=cv,
    n_jobs=1,
    random_state=RANDOM_STATE,
)

t0 = time.time()
rf_search.fit(X_train_tree, y_train, groups=train_groups_arr)
rf_cv_results = (pd.DataFrame(rf_search.cv_results_)
                 .sort_values("rank_test_pr_auc")
                 [["params", "mean_test_pr_auc", "mean_test_roc_auc", "mean_test_f1", "mean_test_accuracy",
                   "mean_test_recall", "mean_test_precision"]])
rf_best_params = {**rf_cv_results.iloc[0]["params"], "n_estimators": 400}
print(f"RF randomised search: {rf_search.n_iter} settings x 5 folds in {time.time() - t0:.1f}s")
print("Best RF params:", rf_best_params)
rf_cv_results.round(4).head(10)

In [ ]:
# Class-weight comparison across the sampled settings
cw = pd.DataFrame(rf_search.cv_results_)
cw["class_weight"] = cw["param_class_weight"].apply(lambda v: "None" if v is None or pd.isna(v) else str(v))
print("Best CV PR-AUC per class_weight option:")
print(cw.groupby("class_weight")[["mean_test_pr_auc", "mean_test_recall", "mean_test_precision"]].max().round(4))

### FINAL MODEL – refit on the full training set

In [ ]:
t0 = time.time()
rf_model = RandomForestClassifier(**rf_best_params, random_state=RANDOM_STATE, n_jobs=N_JOBS).fit(X_train_tree, y_train)
rf_test_proba = rf_model.predict_proba(X_test_tree)[:, 1]
print(f"Final RF fitted on {len(X_train_tree):,} rows in {time.time() - t0:.1f}s")

# Out-of-fold probabilities on the full training set (used only for threshold selection)
t0 = time.time()
rf_oof = oof_rf(rf_best_params, X_train_tree, y_train, train_groups_arr)
print(f"RF out-of-fold predictions on full train in {time.time() - t0:.1f}s")
print("RF OOF metrics @0.5:", {k: round(v, 4) for k, v in evaluate(y_train, rf_oof).items()})

### EFFECT OF TUNING (cross-validated on the full training set)

All three rows are out-of-fold scores on the same rows and folds. "Previous tuned model" is the setting chosen earlier on a 25% sample.

In [ ]:
prev_rf_params = {"n_estimators": 400, "min_samples_split": 2, "min_samples_leaf": 4, "max_features": 0.3,
                  "max_depth": 20, "class_weight": None}
optimisation_steps = pd.DataFrame({
    "0. Baseline (defaults)": rf_baseline_cv,
    "1. Previous tuned model (25% sample)": evaluate(y_train, oof_rf(prev_rf_params, X_train_tree, y_train, train_groups_arr)),
    "2. Retuned on full training set (final)": evaluate(y_train, rf_oof),
}).T
optimisation_steps.round(4)

### FEATURE IMPORTANCE

Impurity importance is fast but biased towards high-cardinality numeric features. Permutation importance is measured on unseen test data as the drop in PR-AUC when a feature is shuffled.

In [ ]:
imp_impurity = pd.Series(rf_model.feature_importances_, index=X_train_tree.columns).sort_values(ascending=False)

perm_sample = X_test_tree.sample(min(5000, len(X_test_tree)), random_state=RANDOM_STATE)
perm = permutation_importance(rf_model, perm_sample, y_test.loc[perm_sample.index],
                              scoring="average_precision", n_repeats=5,
                              random_state=RANDOM_STATE, n_jobs=1)
#   n_jobs=1 here: the forest already predicts on all cores, and copying a large forest
#   to many worker processes can run out of memory
imp_perm = pd.Series(perm.importances_mean, index=X_train_tree.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
imp_impurity.head(15).sort_values().plot(kind="barh", ax=axes[0], color="#2b5c8f")
axes[0].set_title("RF Impurity Importance (Top 15)", fontweight="bold")
imp_perm.head(15).sort_values().plot(kind="barh", ax=axes[1], color="#c0504d")
axes[1].set_title("RF Permutation Importance on Test (drop in PR-AUC)", fontweight="bold")
plt.tight_layout()
plt.show()

### COST-BASED DECISION THRESHOLD

The dataset has no cost figures, so the following **assumption** is stated:

- `COST_FN = 2`: a missed cancellation loses the room revenue for the stay.
- `COST_FP = 1`: a false alarm costs outreach effort, guest friction and a small over-booking risk.

Change these two constants to test other cost ratios. The threshold is chosen on **out-of-fold training predictions** and then applied unchanged to the test set.

In [ ]:
COST_FN = 2.0
COST_FP = 1.0
thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

rows = []
for t in thresholds:
    pred = (rf_oof >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, pred).ravel()
    rows.append({"threshold": t, "cost": COST_FN * fn + COST_FP * fp, "f1": f1_score(y_train, pred)})
thr_table = pd.DataFrame(rows)
rf_threshold = float(thr_table.loc[thr_table["cost"].idxmin(), "threshold"])
f1_thr = float(thr_table.loc[thr_table["f1"].idxmax(), "threshold"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thr_table["threshold"], thr_table["cost"] / len(y_train), color="#2b5c8f", label="Avg cost per booking")
ax.axvline(rf_threshold, color="crimson", ls="--", label=f"Min-cost threshold = {rf_threshold:.2f}")
ax.axvline(0.5, color="grey", ls=":", label="Default 0.50")
ax2 = ax.twinx()
ax2.plot(thr_table["threshold"], thr_table["f1"], color="#2ca02c", alpha=0.6, label=f"F1 (max at {f1_thr:.2f})")
ax2.set_ylabel("F1")
ax.set_title("Random Forest: Cost vs Threshold (OOF train)", fontweight="bold")
ax.set_xlabel("Decision threshold")
ax.set_ylabel(f"Cost (FN={COST_FN:g}, FP={COST_FP:g}) / booking")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="upper center")
plt.tight_layout()
plt.show()
print(f"Chosen RF threshold: {rf_threshold:.2f}")

### TEST-SET EVALUATION

In [ ]:
test_rows = []
for label, t in [("0.50", 0.5), (f"cost-opt {rf_threshold:.2f}", rf_threshold)]:
    tn, fp, fn, tp = confusion_matrix(y_test, (rf_test_proba >= t).astype(int)).ravel()
    test_rows.append({"Threshold": label, **evaluate(y_test, rf_test_proba, t),
                      "Avg cost / booking": (COST_FN * fn + COST_FP * fp) / len(y_test)})
rf_test_results = pd.DataFrame(test_rows).set_index("Threshold")

print(classification_report(y_test, (rf_test_proba >= rf_threshold).astype(int),
                            target_names=["Not canceled", "Canceled"], digits=4))
rf_test_results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, (rf_test_proba >= rf_threshold).astype(int), ax=axes[0], cmap="Blues", colorbar=False,
    display_labels=["Not canceled", "Canceled"])
axes[0].set_title(f"Random Forest – Test Confusion Matrix (t={rf_threshold:.2f})", fontweight="bold")

fpr, tpr, _ = roc_curve(y_test, rf_test_proba)
axes[1].plot(fpr, tpr, color="#2b5c8f", label=f"RF (AUC = {roc_auc_score(y_test, rf_test_proba):.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set(xlabel="False Positive Rate", ylabel="True Positive Rate")
axes[1].set_title("ROC Curve – Test Set", fontweight="bold")
axes[1].legend(loc="lower right")

prec, rec, _ = precision_recall_curve(y_test, rf_test_proba)
axes[2].plot(rec, prec, color="#2b5c8f", label=f"RF (AP = {average_precision_score(y_test, rf_test_proba):.3f})")
axes[2].axhline(y_test.mean(), color="k", ls="--", lw=1, label=f"No-skill ({y_test.mean():.3f})")
axes[2].set(xlabel="Recall", ylabel="Precision")
axes[2].set_title("Precision–Recall Curve – Test Set", fontweight="bold")
axes[2].legend(loc="lower left")
plt.tight_layout()
plt.show()

### TIME-BASED CHECK (train on earlier arrivals, test on later ones)

The tuned forest is refit on training rows arriving **before 2017-01-01** and evaluated on test rows arriving **on or after 2017-01-01**. The encoders from Part B were fitted on the whole training set, so this is an approximate check.

In [ ]:
cutoff = pd.Timestamp("2017-01-01")
early_tr = (meta_train["arrival_date"] < cutoff).to_numpy()
late_te = (meta_test["arrival_date"] >= cutoff).to_numpy()
print(f"Temporal train rows: {early_tr.sum():,} | temporal test rows: {late_te.sum():,}")

rf_time = RandomForestClassifier(**rf_best_params, random_state=RANDOM_STATE, n_jobs=N_JOBS)
rf_time.fit(X_train_tree[early_tr], y_train[early_tr])
rf_time_proba = rf_time.predict_proba(X_test_tree[late_te])[:, 1]

time_results = pd.DataFrame([
    {"Split": "Random (group-aware)", **evaluate(y_test, rf_test_proba, rf_threshold)},
    {"Split": "Temporal (<2017 → 2017)", **evaluate(y_test[late_te], rf_time_proba, rf_threshold)},
]).set_index("Split")
time_results.round(4)

### FAIRNESS CHECK – error rates by country group

`country` is a proxy for nationality. Recall (cancellations caught) and false-positive rate (guests wrongly flagged) are compared across the grouped countries on the test set.

In [ ]:
def group_error_rates(proba, threshold, groups):
    pred = (proba >= threshold).astype(int)
    rows = []
    for g in groups.unique():
        m = (groups == g).to_numpy()
        yt, yp = y_test.to_numpy()[m], pred[m]
        tp = ((yt == 1) & (yp == 1)).sum(); fn = ((yt == 1) & (yp == 0)).sum()
        fp = ((yt == 0) & (yp == 1)).sum(); tn = ((yt == 0) & (yp == 0)).sum()
        rows.append({"country_group": g, "n": m.sum(), "actual_cancel_rate": yt.mean(),
                     "predicted_cancel_rate": yp.mean(),
                     "recall": tp / (tp + fn) if (tp + fn) else np.nan,
                     "FPR": fp / (fp + tn) if (fp + tn) else np.nan})
    return pd.DataFrame(rows).sort_values("n", ascending=False).set_index("country_group")

fair_rf = group_error_rates(rf_test_proba, rf_threshold, meta_test["country_grouped"])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fair_rf["recall"].plot(kind="bar", ax=axes[0], color="#2b5c8f")
axes[0].set_title("RF Recall by Country Group (Test)", fontweight="bold")
fair_rf["FPR"].plot(kind="bar", ax=axes[1], color="#c0504d")
axes[1].set_title("RF False-Positive Rate by Country Group (Test)", fontweight="bold")
for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()
fair_rf.round(3)

### ABLATION – with and without `deposit_type` / `country`

The proposal flags two features for review: the counter-intuitive `deposit_type = Non Refund` pattern (about 99% cancelled) and `country`, which raises fairness concerns. The tuned forest is refit without each feature family to measure how much the model depends on it.

In [ ]:
ablation_sets = {
    "All features": [],
    "Without deposit_type": [c for c in X_train_tree.columns if c.startswith("deposit_type_")],
    "Without country": [c for c in X_train_tree.columns if c.startswith("country_grouped_")],
}
ablation_rows = []
for label, drop in ablation_sets.items():
    if not drop:
        proba = rf_test_proba
    else:
        m = RandomForestClassifier(**rf_best_params, random_state=RANDOM_STATE, n_jobs=N_JOBS)
        m.fit(X_train_tree.drop(columns=drop), y_train)
        proba = m.predict_proba(X_test_tree.drop(columns=drop))[:, 1]
    ablation_rows.append({"Feature set": label, "n_features": X_train_tree.shape[1] - len(drop),
                          **evaluate(y_test, proba, rf_threshold)})
ablation_results = pd.DataFrame(ablation_rows).set_index("Feature set")
ablation_results.round(4)

### SAVE MODEL & RESULTS

In [ ]:
os.makedirs("models", exist_ok=True)
preprocessors = joblib.load("models/preprocessors.joblib")
joblib.dump({"model": rf_model, "params": rf_best_params, "threshold": rf_threshold,
             "features": list(X_train_tree.columns), "preprocessors": preprocessors},
            "models/random_forest_model.joblib", compress=3)   # compressed: a deep forest is large on disk
rf_test_results.to_csv("models/random_forest_test_results.csv")
optimisation_steps.to_csv("models/random_forest_optimisation_steps.csv")

print("Best RF params:", rf_best_params, "| threshold:", rf_threshold)
print("Saved: models/random_forest_model.joblib, models/random_forest_test_results.csv")